In [ ]:
# CELL 1 - Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# CELL 2 - Load Data
df = pd.read_csv('cleaned_retail_1.csv')
print("Shape:", df.shape)
df.info()

In [ ]:
# CELL 3 - Basic EDA
print(df.head())
print("\nNull values:\n", df.isnull().sum())

In [ ]:
# CELL 4 - Filter bad rows
df = df[df["Quantity"] > 0].copy()
df = df[df["UnitPrice"] > 0].copy()
df["TotalAmount"] = df["Quantity"] * df["UnitPrice"]

In [ ]:
# CELL 5 - Parse date (already datetime64 but doing it safely)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df = df.dropna(subset=["InvoiceDate"])  # drop rows where date parse failed

In [ ]:
# CELL 6 - Aggregate to daily sales
daily_sales = (
    df.groupby(df["InvoiceDate"].dt.date)["TotalAmount"]
    .sum()
    .reset_index()
)
daily_sales.columns = ["Date", "Sales"]
daily_sales["Date"] = pd.to_datetime(daily_sales["Date"])
daily_sales = daily_sales.sort_values("Date").reset_index(drop=True)

print("Start:", daily_sales["Date"].min())
print("End  :", daily_sales["Date"].max())
print("Days :", len(daily_sales))

In [ ]:
# CELL 7 - Reindex to fill missing calendar dates
full_dates = pd.date_range(
    start=daily_sales["Date"].min(),
    end=daily_sales["Date"].max(),
    freq="D"
)
daily_sales = (
    daily_sales
    .set_index("Date")
    .reindex(full_dates)
    .rename_axis("Date")
    .reset_index()
)
daily_sales.columns = ["Date", "Sales"]

print("Missing days after reindex:", daily_sales["Sales"].isnull().sum())

In [ ]:
# CELL 8 - Handle missing days properly (NOT fillna(0))
# Forward fill for short gaps (store closed, not zero revenue)
# Then backfill for any leading NaNs
daily_sales["Sales"] = daily_sales["Sales"].fillna(method="ffill").fillna(method="bfill")

print("Nulls after fill:", daily_sales["Sales"].isnull().sum())

In [ ]:
# CELL 9 - Feature Engineering (missing in original notebook)
daily_sales["dayofweek"]  = daily_sales["Date"].dt.dayofweek      # 0=Mon, 6=Sun
daily_sales["month"]      = daily_sales["Date"].dt.month
daily_sales["quarter"]    = daily_sales["Date"].dt.quarter
daily_sales["is_weekend"] = (daily_sales["dayofweek"] >= 5).astype(int)
daily_sales["lag_7"]      = daily_sales["Sales"].shift(7)         # last week same day
daily_sales["lag_14"]     = daily_sales["Sales"].shift(14)
daily_sales["rolling_7"]  = daily_sales["Sales"].shift(1).rolling(7).mean()   # 7-day rolling avg
daily_sales["rolling_30"] = daily_sales["Sales"].shift(1).rolling(30).mean()  # 30-day rolling avg

# Drop NaN rows created by lag features
daily_sales = daily_sales.dropna().reset_index(drop=True)
print("Shape after feature engineering:", daily_sales.shape)

In [ ]:
# CELL 10 - Plot raw time series
plt.figure(figsize=(15, 5))
plt.plot(daily_sales["Date"], daily_sales["Sales"])
plt.title("Daily Sales Time Series")
plt.xlabel("Date")
plt.ylabel("Sales (GBP)")
plt.grid(True)
plt.tight_layout()
plt.show()

print(daily_sales.describe())

# ─────────────────────────────────────────────
# TRAIN / VALIDATION / TEST SPLIT
# Critical: NO random split for time series — always chronological
# ─────────────────────────────────────────────

In [ ]:
# CELL 11 - Split: 70% train | 15% val | 15% test  (approx)
n = len(daily_sales)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

train = daily_sales.iloc[:train_end].copy()
val   = daily_sales.iloc[train_end:val_end].copy()
test  = daily_sales.iloc[val_end:].copy()

print(f"Train : {train['Date'].min().date()} → {train['Date'].max().date()} | {len(train)} days")
print(f"Val   : {val['Date'].min().date()} → {val['Date'].max().date()} | {len(val)} days")
print(f"Test  : {test['Date'].min().date()} → {test['Date'].max().date()} | {len(test)} days")

# ─────────────────────────────────────────────
# PROPHET MODEL
# ─────────────────────────────────────────────

In [ ]:
# CELL 12
# !pip install prophet  ← run this once if not installed

from prophet import Prophet

prophet_train = train[["Date", "Sales"]].rename(columns={"Date": "ds", "Sales": "y"})
prophet_val   = val[["Date", "Sales"]].rename(columns={"Date": "ds", "Sales": "y"})
prophet_test  = test[["Date", "Sales"]].rename(columns={"Date": "ds", "Sales": "y"})

In [ ]:
# CELL 13 - Fit Prophet on TRAIN only
prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05   # regularization — avoids overfitting trend
)
prophet_model.fit(prophet_train)

In [ ]:
# CELL 14 - Validate on VAL set first
val_forecast = prophet_model.predict(prophet_val[["ds"]])
prophet_val_pred = val_forecast["yhat"].values

val_mae  = mean_absolute_error(prophet_val["y"], prophet_val_pred)
val_rmse = np.sqrt(mean_squared_error(prophet_val["y"], prophet_val_pred))
val_smape = 100 * np.mean(
    2 * np.abs(prophet_val_pred - prophet_val["y"].values)
    / (np.abs(prophet_val_pred) + np.abs(prophet_val["y"].values) + 1e-8)
)
print("=== Prophet — Validation Set ===")
print(f"MAE  : {val_mae:.2f}")
print(f"RMSE : {val_rmse:.2f}")
print(f"SMAPE: {val_smape:.2f}%")

In [ ]:
# CELL 15 - Now evaluate on TEST set
test_forecast = prophet_model.predict(prophet_test[["ds"]])
prophet_test_pred = test_forecast["yhat"].values

test_mae  = mean_absolute_error(prophet_test["y"], prophet_test_pred)
test_rmse = np.sqrt(mean_squared_error(prophet_test["y"], prophet_test_pred))
test_smape = 100 * np.mean(
    2 * np.abs(prophet_test_pred - prophet_test["y"].values)
    / (np.abs(prophet_test_pred) + np.abs(prophet_test["y"].values) + 1e-8)
)
print("=== Prophet — Test Set ===")
print(f"MAE  : {test_mae:.2f}")
print(f"RMSE : {test_rmse:.2f}")
print(f"SMAPE: {test_smape:.2f}%")

In [ ]:
# CELL 16 - Plot Prophet test predictions
plt.figure(figsize=(12, 5))
plt.plot(prophet_test["ds"], prophet_test["y"].values, label="Actual")
plt.plot(prophet_test["ds"], prophet_test_pred, label="Prophet Predicted", linestyle="--")
plt.fill_between(
    prophet_test["ds"],
    test_forecast["yhat_lower"].values,
    test_forecast["yhat_upper"].values,
    alpha=0.2, label="Confidence Interval"
)
plt.legend()
plt.title("Prophet — Actual vs Predicted (Test Set)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# CELL 17 - Future forecast (30 days beyond data)
future = prophet_model.make_future_dataframe(periods=30)
future_forecast = prophet_model.predict(future)
prophet_model.plot(future_forecast)
plt.title("Prophet — Future 30 Day Forecast")
plt.show()

future_forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(30).to_csv(
    "future_forecast_prophet.csv", index=False
)

# ─────────────────────────────────────────────
# LSTM MODEL
# ─────────────────────────────────────────────

In [ ]:
# CELL 18 - Scale ONLY on train — NO leakage
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(train[["Sales"]])   # fit ONLY on train

scaled_train = scaler.transform(train[["Sales"]])
scaled_val   = scaler.transform(val[["Sales"]])
scaled_test  = scaler.transform(test[["Sales"]])

print("Scaled shapes — Train:", scaled_train.shape, "| Val:", scaled_val.shape, "| Test:", scaled_test.shape)

In [ ]:
# CELL 19 - Create sequences
def create_sequences(data, look_back=60):
    X, y = [], []
    for i in range(look_back, len(data)):
        X.append(data[i - look_back:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

look_back = 60

# For val sequences: need last 60 days of train as context
train_val_scaled = np.concatenate([scaled_train, scaled_val], axis=0)
train_val_X, train_val_y = create_sequences(train_val_scaled, look_back)

# Train sequences come from pure train
X_train, y_train = create_sequences(scaled_train, look_back)

# Val sequences: only the portion where context is train, label is val
X_val = train_val_X[len(X_train):]
y_val = train_val_y[len(y_train):]

# Test sequences: need last 60 days of val as context
val_test_scaled = np.concatenate([scaled_val, scaled_test], axis=0)
full_X, full_y = create_sequences(val_test_scaled, look_back)
# Test starts after val sequences
# val sequences from val_test_scaled start at index look_back
# we want only the test portion
X_test = full_X[len(scaled_val):]
y_test = full_y[len(scaled_val):]

# Reshape for LSTM: (samples, timesteps, features)
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_val   = X_val.reshape((X_val.shape[0], X_val.shape[1], 1))
X_test  = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

In [ ]:
# CELL 20 - Build LSTM model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

lstm_model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(look_back, 1)),
    Dropout(0.3),
    LSTM(64, return_sequences=False),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dense(1)
])

lstm_model.compile(optimizer="adam", loss="mse")
lstm_model.summary()

In [ ]:
# CELL 21 - Callbacks
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

In [ ]:
# CELL 22 - Train with PROPER val set (not test set)
history = lstm_model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_data=(X_val, y_val),   # ← val, not test
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
# CELL 23 - Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.title("LSTM Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# CELL 24 - Evaluate LSTM on TEST set
test_pred_scaled = lstm_model.predict(X_test)
test_pred = scaler.inverse_transform(test_pred_scaled)
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

lstm_mae  = mean_absolute_error(y_test_actual, test_pred)
lstm_rmse = np.sqrt(mean_squared_error(y_test_actual, test_pred))
lstm_smape = 100 * np.mean(
    2 * np.abs(test_pred - y_test_actual)
    / (np.abs(test_pred) + np.abs(y_test_actual) + 1e-8)
)

print("=== LSTM — Test Set ===")
print(f"MAE  : {lstm_mae:.2f}")
print(f"RMSE : {lstm_rmse:.2f}")
print(f"SMAPE: {lstm_smape:.2f}%")

In [ ]:
# CELL 25 - Plot LSTM test predictions
test_dates = test["Date"].iloc[look_back - len(scaled_val):].reset_index(drop=True)
# Safer: use last len(y_test) dates from test
test_dates = test["Date"].values[-len(y_test_actual):]

plt.figure(figsize=(12, 5))
plt.plot(test_dates, y_test_actual, label="Actual")
plt.plot(test_dates, test_pred, label="LSTM Predicted", linestyle="--")
plt.legend()
plt.title("LSTM — Actual vs Predicted (Test Set)")
plt.grid(True)
plt.tight_layout()
plt.show()

# ─────────────────────────────────────────────
# ENSEMBLE — Weighted combination with validation-optimized weights
# ─────────────────────────────────────────────

In [ ]:
# CELL 26 - Both models must predict on SAME test dates
# Prophet test_pred and LSTM test_pred must align on same dates
# LSTM test covers: test.iloc[look_back:] roughly
# We need to align them carefully

lstm_test_dates = test["Date"].values[-len(y_test_actual):]

# Get prophet predictions for those exact same dates
prophet_aligned = prophet_model.predict(
    pd.DataFrame({"ds": lstm_test_dates})
)
prophet_aligned_pred = prophet_aligned["yhat"].values

print("Prophet aligned:", len(prophet_aligned_pred))
print("LSTM test pred :", len(test_pred.flatten()))
print("Actual         :", len(y_test_actual.flatten()))

In [ ]:
# CELL 27 - Find best weights using VALIDATION SET (not test)
# Get LSTM val predictions
val_pred_scaled = lstm_model.predict(X_val)
val_pred_lstm = scaler.inverse_transform(val_pred_scaled).flatten()
y_val_actual  = scaler.inverse_transform(y_val.reshape(-1, 1)).flatten()

val_dates_lstm = val["Date"].values[-len(y_val_actual):]
prophet_val_aligned = prophet_model.predict(
    pd.DataFrame({"ds": val_dates_lstm})
)
prophet_val_aligned_pred = prophet_val_aligned["yhat"].values

best_smape = np.inf
best_w = 0.5

for w in np.arange(0.0, 1.05, 0.05):
    ensemble_val = w * prophet_val_aligned_pred + (1 - w) * val_pred_lstm
    s = 100 * np.mean(
        2 * np.abs(ensemble_val - y_val_actual)
        / (np.abs(ensemble_val) + np.abs(y_val_actual) + 1e-8)
    )
    if s < best_smape:
        best_smape = s
        best_w = w

print(f"Best Prophet weight (from val): {best_w:.2f}")
print(f"Best LSTM weight              : {1 - best_w:.2f}")
print(f"Validation SMAPE at best w    : {best_smape:.2f}%")

In [ ]:
# CELL 28 - Apply best weights on TEST set
actual_flat      = y_test_actual.flatten()
lstm_pred_flat   = test_pred.flatten()

ensemble_pred = best_w * prophet_aligned_pred + (1 - best_w) * lstm_pred_flat

ens_mae   = mean_absolute_error(actual_flat, ensemble_pred)
ens_rmse  = np.sqrt(mean_squared_error(actual_flat, ensemble_pred))
ens_smape = 100 * np.mean(
    2 * np.abs(ensemble_pred - actual_flat)
    / (np.abs(ensemble_pred) + np.abs(actual_flat) + 1e-8)
)

print("=== Ensemble — Test Set ===")
print(f"MAE  : {ens_mae:.2f}")
print(f"RMSE : {ens_rmse:.2f}")
print(f"SMAPE: {ens_smape:.2f}%")

In [ ]:
# CELL 29 - Final comparison table
print("\n" + "="*45)
print(f"{'Model':<20} {'MAE':>8} {'RMSE':>8} {'SMAPE':>8}")
print("="*45)
print(f"{'Prophet':<20} {test_mae:>8.2f} {test_rmse:>8.2f} {test_smape:>7.2f}%")
print(f"{'LSTM':<20} {lstm_mae:>8.2f} {lstm_rmse:>8.2f} {lstm_smape:>7.2f}%")
print(f"{'Ensemble':<20} {ens_mae:>8.2f} {ens_rmse:>8.2f} {ens_smape:>7.2f}%")
print("="*45)

In [ ]:
# CELL 30 - Final plot: all 3 models vs actual
plt.figure(figsize=(14, 6))
plt.plot(lstm_test_dates, actual_flat,      label="Actual",   linewidth=2)
plt.plot(lstm_test_dates, prophet_aligned_pred, label="Prophet", linestyle="--")
plt.plot(lstm_test_dates, lstm_pred_flat,   label="LSTM",     linestyle="--")
plt.plot(lstm_test_dates, ensemble_pred,    label=f"Ensemble ({best_w:.0%}P + {1-best_w:.0%}L)", linestyle="-.", linewidth=2)
plt.legend()
plt.title("Demand Forecasting — Model Comparison (Test Set)")
plt.xlabel("Date")
plt.ylabel("Sales (GBP)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# CELL 31 - Save models
lstm_model.save("lstm_demand_model.h5")
future_forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(30).to_csv(
    "future_forecast_prophet.csv", index=False
)
print("Models saved.")